# 🚀 Fine-Tuning Qwen3.5 2B for Indian SMS Entity Extraction

**Production-Aligned Interactive Training, Evaluation & Inference Notebook**

This notebook mirrors the production `slm_sms` package (`src/slm_sms/`) end-to-end:
1. **Centralized Config & Observability** (`slm_sms.core`: YAML configs, `settings`, JSON/text logging, OpenTelemetry)
2. **Indian SMS Corpus Synthesis** (`IndianSMSGenerator` with thread-safe RNG, Indian lakh/crore grouping, class-balance guard)
3. **Base Model Inspection & LoRA Configuration** (`slm_sms.models.model_loader`)
4. **Supervised Fine-Tuning** (`SMSTrainingPipeline` with TRL `SFTTrainer` & PEFT LoRA)
5. **Test-Set Benchmarking** (`compute_extraction_metrics` over `data/test.jsonl`)
6. **Single & Batched Inference** (`SMSExtractorEngine.extract` & `extract_batch` with automatic regex fallback)
7. **FastAPI Microservice & OpenTelemetry Verification** (`create_app`, health probes, `X-Request-ID`, `X-Trace-ID`, spans)

### 🛠️ Architecture
```mermaid
flowchart LR
    A["Raw Indian SMS"] --> B["IndianSMSGenerator"]
    B --> C["Chat JSONL Splits (train / val / test)"]
    C --> D["SMSTrainingPipeline (Qwen3.5-2B + LoRA)"]
    D --> E["Trained LoRA Adapter (output/qwen-sms-lora)"]
    E --> F["SMSExtractorEngine (Single + GPU Batch + Fallback)"]
    F --> G["FastAPI + Gunicorn + OpenTelemetry"]
```


## 1. Setup, Configuration, Logging, Telemetry & Hardware Diagnostics
Initialize `slm_sms.core` logging, load `config/generation_config.yaml` and `config/train_config.yaml`, and inspect the available compute device (CUDA / Apple Silicon MPS / CPU).

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project src/ is importable when launched directly from notebooks/
project_root = Path("..").resolve()
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

import datasets
import peft
import pydantic
import torch
import transformers
import trl

from slm_sms.core import configure_logging, configure_telemetry, load_yaml_config, settings
from slm_sms.models.model_loader import get_device_and_dtype

# Readable logs inside Jupyter; OpenTelemetry spans buffered in-memory until configured
configure_logging(level="INFO", use_json=False)
configure_telemetry(exporter_mode="none")

# Load production YAML configs
gen_cfg = load_yaml_config(project_root / "config" / "generation_config.yaml")
train_cfg = load_yaml_config(project_root / "config" / "train_config.yaml")

MODEL_ID = train_cfg["model"]["model_id"]  # Default: Qwen/Qwen3.5-2B
MODEL_REVISION = settings.model_revision  # Optional commit hash pin (SLM_MODEL_REVISION)
DATA_DIR = project_root / "data"
OUTPUT_DIR = project_root / "output" / "qwen-sms-lora"

device, dtype, dev_name = get_device_and_dtype()
print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__} | Transformers: {transformers.__version__}")
print(f"PEFT:         {peft.__version__} | TRL: {trl.__version__}")
print(f"Datasets:     {datasets.__version__} | Pydantic: {pydantic.__version__}")
print(f"Device:       {dev_name} ({device}, dtype={dtype})")
if dev_name == "cuda":
    print(f"GPU Name:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:         {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"Target Model: {MODEL_ID} (revision={MODEL_REVISION or 'latest'})")

## 2. Indian SMS Dataset Generation & Exploration
Uses [`IndianSMSGenerator`](../src/slm_sms/data/generator.py) driven by `config/generation_config.yaml`:
- **Thread-safe seeding**: `random.Random(seed)` isolated per generator instance
- **Indian numbering system**: `_format_indian` formats lakh/crore amounts (`1,45,000.00`, `14,50,000.00`)
- **Class imbalance guard**: Logs class distribution and raises if any SMS class drops below `min_class_fraction` (`2%`)

In [ ]:
import json
from collections import Counter

from slm_sms.data.dataset import save_jsonl
from slm_sms.data.generator import IndianSMSGenerator

ds_params = gen_cfg["dataset"]
generator = IndianSMSGenerator(seed=int(ds_params["seed"]))

data_splits = generator.generate_dataset(
    n_samples=int(ds_params["samples"]),
    val_split=float(ds_params["val_split"]),
    test_split=float(ds_params["test_split"]),
)

print(f"Train split:      {len(data_splits['train'])} samples")
print(f"Validation split: {len(data_splits['validation'])} samples")
print(f"Test split:       {len(data_splits['test'])} samples")

save_jsonl(data_splits["train"], DATA_DIR / "train.jsonl")
save_jsonl(data_splits["validation"], DATA_DIR / "val.jsonl")
save_jsonl(data_splits["test"], DATA_DIR / "test.jsonl")
print(f"Saved JSONL splits to {DATA_DIR}/")

# Inspect class distribution
train_types = Counter(
    json.loads(item["messages"][2]["content"])["type"] for item in data_splits["train"]
)
print("Train type distribution:", dict(train_types))

### Inspect Sample Chat Messages & Indian Lakh Formatting

In [ ]:
print("Indian number format check: Rs.", generator._format_indian(145000.0))
print()

for i in range(4):
    sample = data_splits["train"][i]["messages"]
    print(f"=== Sample {i + 1} ===")
    print(f"System:    {sample[0]['content']}")
    print(f"User SMS:  {sample[1]['content']}")
    print(f"Assistant: {sample[2]['content']}")
    print()

## 3. Base Model Inspection & Zero-Shot Baseline
Run `SMSExtractorEngine` on the **base model** (before fine-tuning) and inspect the chat template.  
If running offline without cached Hugging Face weights, `SMSExtractorEngine` cleanly drops into regex-fallback mode (`fallback_used=True`).

In [ ]:
from slm_sms.inference.engine import SMSExtractorEngine

# Toggle True to load the base model for a zero-shot baseline comparison
RUN_ZERO_SHOT_BASELINE = True

test_sms = (
    "[VM-HDFCBK] Rs.1,450.00 debited from A/c **4821 on 23-11-26 "
    "to swiggy@axisbank (UPI Ref: 432819284129). Avl Bal: Rs.34,210.50."
)

if RUN_ZERO_SHOT_BASELINE:
    base_engine = SMSExtractorEngine(
        model_id_or_path=MODEL_ID,
        model_revision=MODEL_REVISION,
        max_new_tokens=settings.max_new_tokens,
    )
    if base_engine.tokenizer is not None:
        prompt_str = base_engine.tokenizer.apply_chat_template(
            base_engine.build_prompt_messages(test_sms),
            tokenize=False,
            add_generation_prompt=True,
        )
        print("Formatted Qwen chat prompt:\n", prompt_str)

    baseline_result = base_engine.extract(test_sms)
    print("Baseline extraction result:")
    print(json.dumps(baseline_result, indent=2))

## 4. Setup PEFT LoRA (Low-Rank Adaptation)
Pull LoRA hyperparameters directly from `config/train_config.yaml` (`r`, `lora_alpha`, `lora_dropout`, and all Qwen attention + MLP projections).

In [ ]:
from slm_sms.models.model_loader import get_lora_config

lora_cfg_dict = train_cfg["lora"]
lora_config = get_lora_config(
    r=int(lora_cfg_dict["r"]),
    lora_alpha=int(lora_cfg_dict["lora_alpha"]),
    lora_dropout=float(lora_cfg_dict["lora_dropout"]),
    target_modules=lora_cfg_dict.get("target_modules"),
)
print("LoRA Configuration:")
print(lora_config)

## 5. Fine-Tuning with `SMSTrainingPipeline` (SFTTrainer + PEFT + OTel Spans)
Executes the exact same `SMSTrainingPipeline` used by `slm-sms train` / `make train-model`.  
Set `RUN_TRAINING = True` (default) when running on a CUDA GPU or Apple Silicon MPS machine with Hugging Face access / cached weights.

In [ ]:
from slm_sms.data.dataset import load_hf_dataset_from_dir
from slm_sms.training.trainer import SMSTrainingPipeline

RUN_TRAINING = True  # Set False to skip training and test an existing adapter in OUTPUT_DIR

t_cfg = train_cfg["training"]
m_cfg = train_cfg["model"]
hf_dataset = load_hf_dataset_from_dir(DATA_DIR)

pipeline = SMSTrainingPipeline(
    model_id=MODEL_ID,
    output_dir=OUTPUT_DIR,
    lora_r=int(lora_cfg_dict["r"]),
    lora_alpha=int(lora_cfg_dict["lora_alpha"]),
    lora_dropout=float(lora_cfg_dict["lora_dropout"]),
    use_4bit=bool(m_cfg.get("use_4bit", False)),
    use_8bit=bool(m_cfg.get("use_8bit", False)),
)

if RUN_TRAINING:
    train_metrics = pipeline.train(
        dataset=hf_dataset,
        num_train_epochs=int(t_cfg["num_train_epochs"]),
        per_device_train_batch_size=int(t_cfg["per_device_train_batch_size"]),
        gradient_accumulation_steps=int(t_cfg["gradient_accumulation_steps"]),
        learning_rate=float(t_cfg["learning_rate"]),
        max_length=int(t_cfg["max_length"]),
        warmup_ratio=float(t_cfg["warmup_ratio"]),
        logging_steps=int(t_cfg["logging_steps"]),
        eval_steps=t_cfg.get("eval_steps", 50),
        save_total_limit=int(t_cfg["save_total_limit"]),
    )
    print("Training & evaluation metrics:")
    print(json.dumps(train_metrics, indent=2))
else:
    print(f"Skipping training — will load adapter from {OUTPUT_DIR} if present.")

## 6. Test-Set Evaluation & Benchmarking
Runs [`SMSExtractorEngine`](../src/slm_sms/inference/engine.py) with the trained LoRA adapter over `data/test.jsonl` (using native GPU batch inference via `extract_batch`) and scores predictions with `compute_extraction_metrics()`—mirroring `slm-sms evaluate` and `scripts/evaluate.py`.

In [ ]:
from slm_sms.data.dataset import load_jsonl
from slm_sms.training.metrics import compute_extraction_metrics

MAX_EVAL_SAMPLES = 50
test_records = load_jsonl(DATA_DIR / "test.jsonl")[:MAX_EVAL_SAMPLES]

adapter_dir = str(OUTPUT_DIR) if OUTPUT_DIR.exists() else None
eval_engine = SMSExtractorEngine(
    model_id_or_path=MODEL_ID,
    adapter_path=adapter_dir,
    model_revision=MODEL_REVISION,
    max_new_tokens=settings.max_new_tokens,
)

sms_inputs = [rec["messages"][1]["content"] for rec in test_records]
ground_truths = [json.loads(rec["messages"][2]["content"]) for rec in test_records]

# Run batched extraction (automatically uses GPU batching when model is loaded)
batch_results = eval_engine.extract_batch(sms_inputs)
predictions = [json.dumps(r["entity"]) for r in batch_results]
fallback_count = sum(1 for r in batch_results if r["fallback_used"])

report = compute_extraction_metrics(predictions, ground_truths)
print(f"Evaluated {len(test_records)} samples (adapter={adapter_dir}, fallbacks={fallback_count})")
print("\nExtraction Benchmark Report:")
for metric_name, val in report.to_dict().items():
    print(f"  {metric_name:22s}: {val}")

## 7. Interactive Single & Batched Inference (`SMSExtractorEngine`)
Test both `extract()` and `extract_batch()` on diverse Indian SMS messages (UPI debits, OTPs, SBI salary credits, lakh-format amounts, and spam/phishing).

In [ ]:
sample_queries = [
    "[VM-HDFCBK] Rs.1,450.00 debited from A/c **4821 on 23-11-26 to swiggy@axisbank (UPI Ref: 432819284129). Avl Bal: Rs.34,210.50.",
    "489201 is your secret OTP for transaction of Rs.2,499.00 on your Axis Bank Card **7712 at ZOMATO. Valid for 5 mins.",
    "[BZ-SBIINB] A/c *8921 credited by Rs.1,45,000.00 on 23-Nov-26 by Salary transfer from TECH CORP PVT LTD. Avl Bal: Rs.2,87,400.00.",
    "Congratulations! Pre-approved personal loan of Rs.5,00,000 without income proof. Click bit.ly/loan-instant to claim before midnight!",
]

# Single inference example
single_res = eval_engine.extract(sample_queries[0])
print("=== Single Inference ===")
print(json.dumps(single_res, indent=2))

# Batched inference across all 4 queries
print("\n=== Batch Inference ===")
for sms, res in zip(sample_queries, eval_engine.extract_batch(sample_queries)):
    print(f"\nInput SMS: {sms}")
    print(f"Latency: {res['latency_ms']} ms | Fallback used: {res['fallback_used']}")
    print(json.dumps(res["entity"], indent=2))

## 8. FastAPI Microservice & OpenTelemetry Trace Verification
Exercise the production [`create_app()`](../src/slm_sms/inference/server.py) ASGI app in-process with `TestClient` and an OpenTelemetry `InMemorySpanExporter`, verifying:
- `/health/live` & `/health/ready` Kubernetes probes
- `/api/v1/extract` & `/api/v1/extract/batch` HTTP contracts and validation
- `X-Request-ID` & `X-Trace-ID` response headers + parent-child span linkage across the `ThreadPoolExecutor` boundary

In [ ]:
from fastapi.testclient import TestClient
from opentelemetry.sdk.metrics.export import InMemoryMetricReader
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

from slm_sms.core.telemetry import shutdown_telemetry
from slm_sms.inference.server import create_app

os.environ["SLM_LAZY_LOAD"] = "true"
os.environ["SLM_DRAIN_SECONDS"] = "0"

span_exporter = InMemorySpanExporter()
metric_reader = InMemoryMetricReader()
configure_telemetry(
    service_name="slm-sms-notebook",
    span_exporter=span_exporter,
    metric_reader=metric_reader,
    force=True,
)

try:
    notebook_app = create_app(init_telemetry=False)
    with TestClient(notebook_app) as client:
        # Reuse our already initialized eval_engine inside app.state
        client.app.state.engine = eval_engine

        print("Liveness: ", client.get("/health/live").json())
        print("Readiness:", client.get("/health/ready").json())

        resp = client.post(
            "/api/v1/extract",
            json={"text": sample_queries[1]},
            headers={"X-Request-ID": "nb-demo-001"},
        )
        print("\nPOST /api/v1/extract status:", resp.status_code)
        print("X-Request-ID:", resp.headers.get("X-Request-ID"))
        print("X-Trace-ID:  ", resp.headers.get("X-Trace-ID"))
        print("Body:", json.dumps(resp.json(), indent=2))

    emitted_spans = [s.name for s in span_exporter.get_finished_spans()]
    print("\nCaptured OpenTelemetry spans:", emitted_spans)
finally:
    shutdown_telemetry()

## 9. Export Merged Model & Production Deployment
Merge the LoRA adapter into standalone Hugging Face `safetensors` weights for **vLLM**, **Triton**, **Ollama**, or **Gunicorn + FastAPI**:

```bash
# Merge LoRA weights into base model
make export-model

# Launch production Gunicorn + UvicornWorker server
make serve-prod

# Or run with OpenTelemetry OTLP export (e.g. Jaeger / Grafana Collector at http://localhost:4318)
make serve-otel
```
